In [0]:
%run ../../_shared/00_project_config

In [0]:
%run ../../_shared/01_spark_utils

In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window


# ---------------------------------------------------------------------------
# Source and target tables
# ---------------------------------------------------------------------------

CANONICAL_TEAMS_TABLE = (
    f"{SILVER_SCHEMA}.canonical_teams"
)

API_TEAMS_TABLE = (
    f"{SILVER_SCHEMA}.football_data_org_teams"
)

GOLD_TABLE = f"{GOLD_SCHEMA}.dim_team"


# ---------------------------------------------------------------------------
# Business keys and required columns
# ---------------------------------------------------------------------------

BUSINESS_KEY = [
    "team_key",
]

REQUIRED_GOLD_COLUMNS = [
    "team_key",
    "team_name",
]


# ---------------------------------------------------------------------------
# Validate source tables
# ---------------------------------------------------------------------------

for table_name in [
    CANONICAL_TEAMS_TABLE,
    API_TEAMS_TABLE,
]:
    if not spark.catalog.tableExists(table_name):
        raise ValueError(
            f"Required Silver table does not exist: {table_name}"
        )

canonical_raw_df = spark.table(
    CANONICAL_TEAMS_TABLE
)

api_raw_df = spark.table(
    API_TEAMS_TABLE
)

validate_columns_exist(
    canonical_raw_df,
    [
        "team_key",
        "team_name",
    ],
)

validate_columns_exist(
    api_raw_df,
    [
        "team_key",
        "team_id",
        "season_start_year",
        "league_code",
        "team_short_name",
        "team_tla",
        "area_name",
        "founded_year",
        "venue",
        "website",
        "team_crest",
        "club_colors",
        "last_updated",
        "_silver_processed_at",
    ],
)


# ---------------------------------------------------------------------------
# Prepare canonical team dimension base
# ---------------------------------------------------------------------------

canonical_df = (
    canonical_raw_df
    .select(
        F.col("team_key")
        .cast("bigint")
        .alias("team_key"),

        F.trim(
            F.col("team_name")
        ).alias("team_name"),
    )
)

canonical_metrics = validate_required_not_null(
    canonical_df,
    REQUIRED_GOLD_COLUMNS,
    fail_on_null=True,
)

if canonical_metrics["row_count"] == 0:
    raise ValueError(
        "Canonical teams dataset is empty. "
        "Refusing to build dim_team."
    )

blank_names = (
    canonical_df
    .filter(
        F.length(F.col("team_name")) == 0
    )
    .limit(1)
    .count()
)

if blank_names > 0:
    raise ValueError(
        "Canonical team names cannot be blank."
    )

duplicate_canonical_keys = (
    canonical_df
    .groupBy("team_key")
    .count()
    .filter(F.col("count") > 1)
)

if duplicate_canonical_keys.limit(1).count() > 0:
    display(duplicate_canonical_keys)

    raise ValueError(
        "Duplicate team_key values found in canonical_teams."
    )

print(
    "Canonical team base validated: "
    f"{canonical_metrics['row_count']} teams."
)


# ---------------------------------------------------------------------------
# Prepare API team enrichment
# ---------------------------------------------------------------------------

# The API source contains team records at the competition-season
# level. Select one deterministic descriptive record per team_key.

api_prepared_df = (
    api_raw_df
    .select(
        F.col("team_key")
        .cast("bigint")
        .alias("team_key"),

        F.col("team_id")
        .cast("int")
        .alias("api_team_id"),

        F.col("season_start_year")
        .cast("int")
        .alias("season_start_year"),

        F.col("league_code"),

        F.col("team_short_name"),
        F.col("team_tla"),

        F.col("area_name")
        .alias("country"),

        F.col("founded_year")
        .cast("int")
        .alias("founded_year"),

        F.col("venue"),
        F.col("website"),
        F.col("team_crest"),
        F.col("club_colors"),
        F.col("last_updated"),
        F.col("_silver_processed_at"),
    )
)

validate_required_not_null(
    api_prepared_df,
    ["team_key"],
    fail_on_null=True,
)

unmapped_api_teams = (
    api_prepared_df
    .select("team_key")
    .distinct()
    .join(
        canonical_df.select("team_key"),
        on="team_key",
        how="left_anti",
    )
)

if unmapped_api_teams.limit(1).count() > 0:
    display(unmapped_api_teams)

    raise ValueError(
        "API teams contain unknown canonical team_key values."
    )


# ---------------------------------------------------------------------------
# Select latest API attributes for each team
# ---------------------------------------------------------------------------

attribute_columns = [
    "team_short_name",
    "team_tla",
    "country",
    "founded_year",
    "venue",
    "website",
    "team_crest",
    "club_colors",
]

api_prepared_df = api_prepared_df.withColumn(
    "_attribute_tiebreaker",
    F.to_json(
        F.struct(
            *[
                F.col(column_name).alias(column_name)
                for column_name in attribute_columns
            ]
        ),
        options={"ignoreNullFields": "false"},
    ),
)

team_window = (
    Window
    .partitionBy("team_key")
    .orderBy(
        F.col("season_start_year").desc_nulls_last(),
        F.col("last_updated").desc_nulls_last(),
        F.col("_silver_processed_at").desc_nulls_last(),
        F.col("api_team_id").desc_nulls_last(),
        F.col("league_code").asc_nulls_last(),
        F.col("_attribute_tiebreaker").asc_nulls_last(),
    )
)

api_latest_df = (
    api_prepared_df
    .withColumn(
        "_row_number",
        F.row_number().over(team_window),
    )
    .filter(F.col("_row_number") == 1)
    .select(
        "team_key",
        *attribute_columns,
    )
)

duplicate_api_keys = (
    api_latest_df
    .groupBy("team_key")
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
)

if duplicate_api_keys > 0:
    raise ValueError(
        "API enrichment contains duplicate team keys."
    )

print("API team enrichment prepared.")


# ---------------------------------------------------------------------------
# Build Gold team dimension
# ---------------------------------------------------------------------------

gold_df = (
    canonical_df
    .join(
        api_latest_df,
        on="team_key",
        how="left",
    )
    .select(
        F.col("team_key"),
        F.col("team_name"),
        F.col("team_short_name"),
        F.col("team_tla"),
        F.col("country"),
        F.col("founded_year"),
        F.col("venue"),
        F.col("website"),
        F.col("team_crest"),
        F.col("club_colors"),
    )
    .withColumn(
        "_gold_processed_at",
        F.current_timestamp(),
    )
)


# ---------------------------------------------------------------------------
# Validate Gold team dimension
# ---------------------------------------------------------------------------

gold_metrics = validate_required_not_null(
    gold_df,
    REQUIRED_GOLD_COLUMNS,
    fail_on_null=True,
)

gold_count = gold_metrics["row_count"]
canonical_count = canonical_metrics["row_count"]

if gold_count != canonical_count:
    raise ValueError(
        "Gold team count does not match canonical team count: "
        f"gold={gold_count}, canonical={canonical_count}"
    )

duplicate_gold_keys = (
    gold_df
    .groupBy(*BUSINESS_KEY)
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
)

if duplicate_gold_keys > 0:
    raise ValueError(
        "Duplicate team_key values found in dim_team."
    )

gold_team_keys = gold_df.select("team_key")

missing_gold_teams = (
    canonical_df
    .select("team_key")
    .join(
        gold_team_keys,
        on="team_key",
        how="left_anti",
    )
    .limit(1)
    .count()
)

if missing_gold_teams > 0:
    raise ValueError(
        "Some canonical teams are missing from dim_team."
    )

invalid_founded_year = (
    gold_df
    .filter(
        F.col("founded_year").isNotNull()
        & (
            (F.col("founded_year") < 1800)
            | (
                F.col("founded_year")
                > F.year(F.current_date())
            )
        )
    )
    .limit(1)
    .count()
)

if invalid_founded_year > 0:
    raise ValueError(
        "Invalid founded_year detected in dim_team."
    )

print(
    "Gold validation successful: "
    f"{gold_count} unique canonical teams."
)


# ---------------------------------------------------------------------------
# Write Gold Delta table
# ---------------------------------------------------------------------------

# Full refresh is appropriate for this small dimension.
# Repeated executions replace existing records instead of appending.

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}"
)

write_full_refresh(
    gold_df,
    GOLD_TABLE,
)


# ---------------------------------------------------------------------------
# Post-write validation
# ---------------------------------------------------------------------------

saved_df = spark.table(GOLD_TABLE)

saved_count = saved_df.count()

if saved_count != canonical_count:
    raise ValueError(
        "Post-write validation failed: "
        f"expected {canonical_count} rows, "
        f"found {saved_count}."
    )

saved_duplicate_keys = (
    saved_df
    .groupBy("team_key")
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
)

if saved_duplicate_keys > 0:
    raise ValueError(
        "Post-write validation failed: duplicate team_key values."
    )

enriched_count = (
    saved_df
    .filter(
        F.col("team_short_name").isNotNull()
    )
    .count()
)


# ---------------------------------------------------------------------------
# Display execution summary
# ---------------------------------------------------------------------------

print("=" * 55)
print("GOLD DIMENSION LOAD COMPLETED")
print("=" * 55)
print(f"Target table: {GOLD_TABLE}")
print(f"Total canonical teams: {saved_count}")
print(f"Teams with API short names: {enriched_count}")
print(
    "Teams without API short names: "
    f"{saved_count - enriched_count}"
)
print("=" * 55)

display(
    saved_df.orderBy("team_name")
)
